In [2]:
from dataclasses import dataclass
from itertools import product
from pathlib import Path
import json
import math
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

try:
    import peptides
    from peptides import Peptide
except ImportError as exc:
    raise ImportError(
        "This notebook requires peptides.py. Install it in the active environment: pip install peptides"
    ) from exc

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 190)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "matrices" / "NylC_Puetz_raw_data.CSV").exists():
            return candidate
    if start.name == "Notebooks":
        return start.parent
    raise FileNotFoundError("Could not locate matrices/NylC_Puetz_raw_data.CSV")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "matrices" / "NylC_Puetz_raw_data.CSV"
WT_FASTA = PROJECT_ROOT / "inputs" / "variant_fastas" / "WT.fasta"
BOLTZ_CANDIDATE_RELATIVE_PATH = Path("inputs") / "boltz_variants" / "final_designs_metrics_160.csv"
BOLTZ_CANDIDATE_PATH = PROJECT_ROOT / BOLTZ_CANDIDATE_RELATIVE_PATH
BOLTZ_DIR = BOLTZ_CANDIDATE_PATH.parent
ANOVA_OUTPUT_DIR = PROJECT_ROOT / "results" / "anova_gp_reviewed"
OUT_DIR = PROJECT_ROOT / "results" / "paretro_selection_updated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", PROJECT_ROOT)
print("Output:", OUT_DIR)


Python: 3.12.3
Platform: Windows-11-10.0.26200-SP0
Project root: c:\Users\arthu\OneDrive\Dokumente\Bachelorarbeit\nylon_md_features
Output: c:\Users\arthu\OneDrive\Dokumente\Bachelorarbeit\nylon_md_features\results\paretro_selection_updated


In [3]:
CANONICAL_AA = tuple("ACDEFGHIKLMNPQRSTVWY")
WT_POCKET = {99: "D", 134: "F", 304: "D", 330: "R"}
POSITIONS = tuple(WT_POCKET.keys())
TARGET_POSITIONS = set(POSITIONS)

In [4]:
try:
    import peptides
    from peptides import Peptide
except ImportError as exc:
    raise ImportError(
        "This notebook requires peptides.py. Install it in the active environment: pip install peptides"
    ) from exc


def read_fasta_sequence_local(path):
    sequence = "".join(
        line.strip()
        for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.startswith(">")
    ).upper()
    if not sequence:
        raise ValueError(f"No sequence found in {path}")
    return sequence


WT_SEQUENCE = read_fasta_sequence_local(WT_FASTA)
df_raw = pd.read_csv(DATA_PATH, sep=";", decimal=",")
activity_rep_cols = [col for col in df_raw.columns if col.startswith("activity_pa6_")]
tm_rep_cols = [col for col in df_raw.columns if col.startswith("tm_celsius_")]
if not activity_rep_cols:
    raise ValueError("No PA6 activity replicate columns found.")

df = df_raw.copy()
df[activity_rep_cols] = df[activity_rep_cols].apply(pd.to_numeric, errors="coerce")
if tm_rep_cols:
    df[tm_rep_cols] = df[tm_rep_cols].apply(pd.to_numeric, errors="coerce")
df["activity_n"] = df[activity_rep_cols].count(axis=1)
df["activity_pa6"] = df[activity_rep_cols].mean(axis=1, skipna=True)
df["activity_sd"] = df[activity_rep_cols].std(axis=1, skipna=True, ddof=1)
df["activity_sem"] = df["activity_sd"] / np.sqrt(df["activity_n"])
df["tm_celsius"] = df[tm_rep_cols].mean(axis=1, skipna=True) if tm_rep_cols else np.nan
if df["activity_pa6"].isna().any():
    raise ValueError("At least one lab variant has no usable PA6 activity.")

fallback_sem = df.loc[df["activity_sem"].notna() & (df["activity_sem"] > 0), "activity_sem"].median()
if not np.isfinite(fallback_sem):
    fallback_sem = 0.0
df["activity_sem_for_gp"] = df["activity_sem"].fillna(fallback_sem)

MUTATION_PATTERN = re.compile(r"([A-Z])(\d+)([A-Z])")


def parse_mutations(mutation_string):
    pocket = dict(WT_POCKET)
    if pd.isna(mutation_string) or str(mutation_string).strip().lower() in {
        "", "wt", "wildtype", "wild type", "nan",
    }:
        return pocket, []
    mutations = []
    for wt_aa, pos_str, mut_aa in MUTATION_PATTERN.findall(str(mutation_string).upper()):
        pos = int(pos_str)
        mutations.append((wt_aa, pos, mut_aa))
        if pos in WT_POCKET:
            expected = WT_POCKET[pos]
            if wt_aa != expected:
                warnings.warn(
                    f"Mutation {wt_aa}{pos}{mut_aa} does not match expected WT {expected}{pos}.",
                    RuntimeWarning,
                )
            pocket[pos] = mut_aa
    return pocket, mutations


def mutation_positions(mutations):
    return {pos for _, pos, _ in mutations}


def mutation_signature(positions):
    return "WT" if not positions else "+".join(str(pos) for pos in sorted(positions))


def mutation_list_to_string(mutations):
    return "" if not mutations else ";".join(
        f"{wt}{pos}{mut}" for wt, pos, mut in sorted(mutations, key=lambda item: item[1])
    )


parsed = df["mutations"].apply(parse_mutations)
df_model = df.copy()
df_model["pocket"] = parsed.apply(lambda item: item[0])
df_model["parsed_mutations"] = parsed.apply(lambda item: item[1])
df_model["all_mutations"] = df_model["parsed_mutations"]
df_model["mutation_positions"] = df_model["all_mutations"].apply(mutation_positions)
df_model["mutation_signature"] = df_model["mutation_positions"].apply(mutation_signature)
df_model["mutation_order"] = df_model["all_mutations"].apply(len)
df_model["aa_tuple"] = df_model["pocket"].apply(lambda pocket: tuple(pocket[pos] for pos in POSITIONS))
df_model["only_target_positions"] = df_model["mutation_positions"].apply(
    lambda positions: positions.issubset(TARGET_POSITIONS)
)
df_model["mutation_set"] = df_model["all_mutations"].apply(
    lambda muts: frozenset(f"{wt}{pos}{mut}" for wt, pos, mut in muts)
)

df_core = df_model[df_model["only_target_positions"]].copy().reset_index(drop=True)
df_excluded = df_model[~df_model["only_target_positions"]].copy()
df_lab = df
df_lab_model = df_model
df_train_gp = df_core

if len(df_core) != df_core["variant_id"].nunique():
    raise ValueError("Core training variant IDs must be unique.")
if len(df_core) < 3:
    raise ValueError("At least three eligible lab variants are required.")

print("WT length:", len(WT_SEQUENCE))
print("Lab variants:", len(df_model))
print("Four-position GP training variants:", len(df_core))
print("Excluded because of external mutations:", len(df_excluded))
display(df_core[["variant_id", "mutations", "aa_tuple", "activity_pa6", "activity_sem_for_gp"]].head(12))


WT length: 355
Lab variants: 36
Four-position GP training variants: 35
Excluded because of external mutations: 1


,variant_id,mutations,aa_tuple,activity_pa6,activity_sem_for_gp
0,WT,NaN,"(D, F, D, R)",77.666667,1.333333
1,D99G,D99G,"(G, F, D, R)",144.000000,11.150486
2,D99V,D99V,"(V, F, D, R)",147.666667,3.480102
3,D99R,D99R,"(R, F, D, R)",194.000000,10.000000
4,F134W,F134W,"(D, W, D, R)",217.000000,5.000000
5,D304M,D304M,"(D, F, M, R)",233.500000,2.500000
6,D304E,D304E,"(D, F, E, R)",171.250000,4.516175
7,D304Q,D304Q,"(D, F, Q, R)",157.000000,6.429101
8,D304V,D304V,"(D, F, V, R)",131.500000,13.500000
9,D304W,D304W,"(D, F, W, R)",130.000000,5.000000


In [5]:
BOLTZGEN_POSITION_MAP = [
    ("full_sequence_0", 82, 99),
    ("full_sequence_0", 117, 134),
    ("full_sequence_3", 38, 304),
    ("full_sequence_7", 64, 330),
]

# Relative to PROJECT_ROOT (= nylon_md_features). The exact input is
# inputs/boltz_variants/final_designs_metrics_160.csv.
CANDIDATE_CSV_PATTERN = BOLTZ_CANDIDATE_PATH.name


def load_boltzgen_candidate_csvs(candidate_dir=BOLTZ_DIR, csv_pattern=CANDIDATE_CSV_PATTERN):
    candidate_dir = Path(candidate_dir)
    if not candidate_dir.exists():
        raise FileNotFoundError(f"Candidate directory does not exist: {candidate_dir}")

    csv_files = sorted(candidate_dir.glob(csv_pattern))
    if len(csv_files) == 0:
        available_csvs = sorted(candidate_dir.rglob("*.csv"))
        msg = (
            f"No candidate CSV files found in {candidate_dir} "
            f"with pattern {csv_pattern}.\n\nAvailable CSV files:\n"
        )
        msg += "\n".join(str(file) for file in available_csvs[:50]) if available_csvs else "No CSV files found at all."
        raise FileNotFoundError(msg)

    print("Loaded candidate CSV files:")
    dfs = []
    for csv_file in csv_files:
        print(" ", csv_file)
        tmp = pd.read_csv(csv_file)
        tmp["source_file"] = csv_file.name
        tmp["source_path"] = str(csv_file)
        tmp["boltzgen_run"] = csv_file.stem.replace("final_designs_metrics_", "")
        dfs.append(tmp)

    return pd.concat(dfs, ignore_index=True, sort=False)


def validate_required_boltzgen_columns(df, position_map=BOLTZGEN_POSITION_MAP):
    required_cols = sorted(set(col for col, _, _ in position_map))
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError("Missing required BoltzGen sequence columns: " + ", ".join(missing_cols))

    print("Required sequence columns found:")
    for col in required_cols:
        lengths = df[col].dropna().astype(str).str.len()
        print(
            col,
            "min_len:", lengths.min(),
            "max_len:", lengths.max(),
            "unique_sequences:", df[col].nunique(),
        )


def get_aa_from_chain_sequence(row, sequence_col, local_pos):
    if sequence_col not in row.index:
        return None, f"missing_column_{sequence_col}"
    seq = row[sequence_col]
    if pd.isna(seq):
        return None, f"missing_value_{sequence_col}"

    seq = str(seq).strip().upper()
    if len(seq) < local_pos:
        return None, f"{sequence_col}_too_short_for_position_{local_pos}"

    aa = seq[local_pos - 1]
    if aa not in CANONICAL_AA:
        return None, f"noncanonical_{aa}_in_{sequence_col}_at_{local_pos}"

    return aa, "ok"


def extract_boltzgen_pocket_from_chain_sequences(row, position_map=BOLTZGEN_POSITION_MAP, wt_pocket=WT_POCKET):
    pocket = {}
    mutations = []
    status_messages = []

    for sequence_col, local_pos, model_pos in position_map:
        aa, status = get_aa_from_chain_sequence(row, sequence_col, local_pos)
        if status != "ok":
            status_messages.append(status)
            continue

        pocket[model_pos] = aa
        wt_aa = wt_pocket[model_pos]
        if aa != wt_aa:
            mutations.append((wt_aa, model_pos, aa))

    expected_positions = set(POSITIONS)
    if set(pocket.keys()) != expected_positions:
        return None, "incomplete_pocket", None, None

    if status_messages:
        return None, ";".join(status_messages), None, None

    mutated_positions = tuple(pos for _, pos, _ in mutations)
    return pocket, "ok", mutations, mutated_positions


def add_boltzgen_candidates_from_full_sequences(df):
    df = df.copy()
    validate_required_boltzgen_columns(df, BOLTZGEN_POSITION_MAP)

    if "id" in df.columns:
        df["candidate_id"] = df["boltzgen_run"].astype(str) + "__" + df["id"].astype(str)
    else:
        df["candidate_id"] = [f"candidate_{i:05d}" for i in range(len(df))]

    pockets = []
    statuses = []
    mutations = []
    mutation_strings = []
    mutated_positions = []

    for _, row in df.iterrows():
        pocket, status, mutation_list, pos_tuple = extract_boltzgen_pocket_from_chain_sequences(row)
        pockets.append(pocket)
        statuses.append(status)
        mutations.append(mutation_list if mutation_list is not None else [])
        mutation_strings.append(mutation_list_to_string(mutation_list or []))
        mutated_positions.append(pos_tuple)

    df["pocket"] = pockets
    df["candidate_status"] = statuses
    df["parsed_mutations"] = mutations
    df["mutations"] = mutation_strings
    df["mutated_positions"] = mutated_positions

    df_valid = df[df["candidate_status"].eq("ok")].copy()
    df_valid["aa_tuple"] = df_valid["pocket"].apply(lambda pocket: tuple(pocket[pos] for pos in POSITIONS))
    for pos in POSITIONS:
        df_valid[f"aa{pos}"] = df_valid["pocket"].apply(lambda pocket, pos=pos: pocket[pos])

    df_valid["n_target_mutations"] = df_valid["parsed_mutations"].apply(len)
    df_valid["n_mutations"] = df_valid["n_target_mutations"]
    df_valid["mutation_positions"] = df_valid["parsed_mutations"].apply(mutation_positions)
    df_valid["mutation_signature"] = df_valid["mutation_positions"].apply(mutation_signature)
    df_valid["mutation_set"] = df_valid["parsed_mutations"].apply(
        lambda muts: frozenset(f"{wt}{pos}{mut}" for wt, pos, mut in muts)
    )
    df_valid["only_gp_target_positions"] = True
    df_valid["n_external_mutations"] = 0

    # Deduplicate identical four-position pockets across BoltzGen runs, keeping
    # the structurally strongest representative.
    sort_cols = [col for col in ["quality_score", "design_to_target_iptm", "design_ptm"] if col in df_valid.columns]
    if sort_cols:
        df_valid = df_valid.sort_values(sort_cols, ascending=[False] * len(sort_cols))
    df_valid = df_valid.drop_duplicates(subset=["aa_tuple"], keep="first").reset_index(drop=True)

    return df, df_valid


def check_designed_sequence_consistency(df_valid, strict=True):
    if "designed_sequence" not in df_valid.columns:
        print("Column designed_sequence not found. Skipping consistency check.")
        return df_valid

    df_check = df_valid.copy()
    df_check["designed_sequence_clean"] = df_check["designed_sequence"].astype(str).str.strip().str.upper()
    df_check["extracted_designed_sequence"] = (
        df_check["aa99"] + df_check["aa134"] + df_check["aa304"] + df_check["aa330"]
    )
    df_check["designed_sequence_matches"] = (
        df_check["designed_sequence_clean"] == df_check["extracted_designed_sequence"]
    )

    match_fraction = df_check["designed_sequence_matches"].mean()
    n_mismatches = int((~df_check["designed_sequence_matches"]).sum())
    print("\nDesigned-sequence consistency:")
    print("Expected order: aa99 + aa134 + aa304 + aa330")
    print("Match fraction:", match_fraction)
    print("Number of mismatches:", n_mismatches)

    display(
        df_check[
            [
                "candidate_id",
                "designed_sequence_clean",
                "extracted_designed_sequence",
                "designed_sequence_matches",
                "mutations",
                "aa99",
                "aa134",
                "aa304",
                "aa330",
            ]
        ].head(30)
    )

    if strict and n_mismatches > 0:
        display(
            df_check.loc[
                ~df_check["designed_sequence_matches"],
                [
                    "candidate_id",
                    "designed_sequence_clean",
                    "extracted_designed_sequence",
                    "mutations",
                    "aa99",
                    "aa134",
                    "aa304",
                    "aa330",
                    "source_file",
                ],
            ]
        )
        raise ValueError(
            "designed_sequence does not match extracted amino acids. "
            "Check BOLTZGEN_POSITION_MAP or chain numbering."
        )

    return df_check


boltzgen_candidates_raw = load_boltzgen_candidate_csvs(
    candidate_dir=BOLTZ_DIR,
    csv_pattern=CANDIDATE_CSV_PATTERN,
)
boltzgen_candidates_all, boltzgen_candidates_valid = add_boltzgen_candidates_from_full_sequences(
    boltzgen_candidates_raw
)
boltzgen_candidates_valid_checked = check_designed_sequence_consistency(
    boltzgen_candidates_valid,
    strict=True,
)

candidates = boltzgen_candidates_valid_checked.copy()

tested_mutation_sets = set(df_lab_model["mutation_set"])
candidates["already_tested"] = candidates["mutation_set"].isin(tested_mutation_sets)

print("\nRaw BoltzGen candidates:", boltzgen_candidates_raw.shape)
print("Candidates after extraction:", boltzgen_candidates_all.shape)
print("Valid deduplicated ANOVA-GP candidates:", candidates.shape)
print("\nCandidate status counts:")
print(boltzgen_candidates_all["candidate_status"].value_counts(dropna=False))
print("\nMutation count distribution:")
print(candidates["n_target_mutations"].value_counts(dropna=False).sort_index())

display(
    candidates[
        [
            "candidate_id",
            "mutations",
            "mutated_positions",
            "aa99",
            "aa134",
            "aa304",
            "aa330",
            "n_target_mutations",
            "source_file",
        ]
    ].head(30)
)

Loaded candidate CSV files:
  c:\Users\arthu\OneDrive\Dokumente\Bachelorarbeit\nylon_md_features\inputs\boltz_variants\final_designs_metrics_160.csv
Required sequence columns found:
full_sequence_0 min_len: 243 max_len: 243 unique_sequences: 47
full_sequence_3 min_len: 89 max_len: 89 unique_sequences: 6
full_sequence_7 min_len: 89 max_len: 89 unique_sequences: 8

Designed-sequence consistency:
Expected order: aa99 + aa134 + aa304 + aa330
Match fraction: 1.0
Number of mismatches: 0


,candidate_id,designed_sequence_clean,extracted_designed_sequence,designed_sequence_matches,mutations,aa99,aa134,aa304,aa330
0,160__task-1372632-1_NylC_paper_48,DPLG,DPLG,True,F134P;D304L;R330G,D,P,L,G
1,160__task-1372632-25_NylC_paper_21,SFLA,SFLA,True,D99S;D304L;R330A,S,F,L,A
2,160__task-1372632-105_NylC_paper_15,DFLG,DFLG,True,D304L;R330G,D,F,L,G
3,160__task-1372632-10_NylC_paper_18,SILG,SILG,True,D99S;F134I;D304L;R330G,S,I,L,G
4,160__task-1372632-104_NylC_paper_01,DFLS,DFLS,True,D304L;R330S,D,F,L,S
5,160__task-1372632-91_NylC_paper_06,DVLS,DVLS,True,F134V;D304L;R330S,D,V,L,S
6,160__task-1372632-132_NylC_paper_10,AFLG,AFLG,True,D99A;D304L;R330G,A,F,L,G
7,160__task-1372632-40_NylC_paper_48,EFLS,EFLS,True,D99E;D304L;R330S,E,F,L,S
8,160__task-1372632-103_NylC_paper_42,NPLG,NPLG,True,D99N;F134P;D304L;R330G,N,P,L,G
9,160__task-1372632-22_NylC_paper_45,GPLN,GPLN,True,D99G;F134P;D304L;R330N,G,P,L,N



Raw BoltzGen candidates: (160, 217)
Candidates after extraction: (160, 223)
Valid deduplicated ANOVA-GP candidates: (160, 239)

Candidate status counts:
candidate_status
ok    160
Name: count, dtype: int64

Mutation count distribution:
n_target_mutations
2    11
3    92
4    57
Name: count, dtype: int64


,candidate_id,mutations,mutated_positions,aa99,aa134,aa304,aa330,n_target_mutations,source_file
0,160__task-1372632-1_NylC_paper_48,F134P;D304L;R330G,"(134, 304, 330)",D,P,L,G,3,final_designs_metrics_160.csv
1,160__task-1372632-25_NylC_paper_21,D99S;D304L;R330A,"(99, 304, 330)",S,F,L,A,3,final_designs_metrics_160.csv
2,160__task-1372632-105_NylC_paper_15,D304L;R330G,"(304, 330)",D,F,L,G,2,final_designs_metrics_160.csv
3,160__task-1372632-10_NylC_paper_18,D99S;F134I;D304L;R330G,"(99, 134, 304, 330)",S,I,L,G,4,final_designs_metrics_160.csv
4,160__task-1372632-104_NylC_paper_01,D304L;R330S,"(304, 330)",D,F,L,S,2,final_designs_metrics_160.csv
5,160__task-1372632-91_NylC_paper_06,F134V;D304L;R330S,"(134, 304, 330)",D,V,L,S,3,final_designs_metrics_160.csv
6,160__task-1372632-132_NylC_paper_10,D99A;D304L;R330G,"(99, 304, 330)",A,F,L,G,3,final_designs_metrics_160.csv
7,160__task-1372632-40_NylC_paper_48,D99E;D304L;R330S,"(99, 304, 330)",E,F,L,S,3,final_designs_metrics_160.csv
8,160__task-1372632-103_NylC_paper_42,D99N;F134P;D304L;R330G,"(99, 134, 304, 330)",N,P,L,G,4,final_designs_metrics_160.csv
9,160__task-1372632-22_NylC_paper_45,D99G;F134P;D304L;R330N,"(99, 134, 304, 330)",G,P,L,N,4,final_designs_metrics_160.csv


In [6]:
print(boltzgen_candidates_raw["design_largest_hydrophobic_patch_refolded"])

0      3892.94141
1      3942.27856
2      3957.90869
3      4011.63135
4      4034.37036
          ...    
155    7458.41211
156    7479.65967
157    3855.94434
158    7527.14355
159    7382.47656
Name: design_largest_hydrophobic_patch_refolded, Length: 160, dtype: float64
